In [ ]:
from MultiAgent_DQ import *
import numpy as np
import pandas as pd  
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
import sys
sys.path.append('../')
import tqdm
from utils import *

## Proposition17


In [ ]:
from basin import (
    calculate_B_hat,
    calculate_C_hat,
    calculate_D_hat,
    get_aggregate_bounds_prop17,
    theoretical_P_AD_from_ADGT,
    plot_adgt_boa_validation,
    plot_acgt_boundary_validation,
    plot_acad_entry_validation,
    sample_acad_entry_points,
    verify_trajectory_in_basin,
)
from MultiAgent_DQ import Algorithm
from utils import convert_dict_to_mean_df


def theoretical_P_GT_from_ADGT(t, r, p, s, epsilon=None, delta=None, state_idx=0, e=None):
  """B_hat = BoA(pi^GT | (AD,GT))."""
  if epsilon is None:
    epsilon = e
  return calculate_B_hat(t=t, r=r, p=p, s=s, epsilon=epsilon, d=delta, state_idx=state_idx)


def theoretical_C_hat_from_ACGT(t, r, p, s, epsilon=None, delta=None, e=None):
  if epsilon is None:
    epsilon = e
  return calculate_C_hat(t=t, r=r, p=p, s=s, epsilon=epsilon, d=delta)


def theoretical_D_hat_from_ACAD(t, r, p, s, epsilon=None, delta=None, e=None):
  if epsilon is None:
    epsilon = e
  return calculate_D_hat(t=t, r=r, p=p, s=s, epsilon=epsilon, d=delta)



## reading results from simulations


In [ ]:
df = pd.read_csv("../results/results_from_vectorized_script_raw.csv")
df_08_05 = pd.read_csv('../results/results_vect_ADGT_0.8_0.5_more_e.csv')
df_07_02 = pd.read_csv('../results/results_vect_ADGT_0.7_0.2_more_e.csv')
df_08_05_uniform = pd.read_csv('../results/results_vect_uniform_0.8_0.5_more_e.csv')
df_07_02_uniform = pd.read_csv('../results/results_vect_uniform_0.7_0.2_more_e.csv')



In [ ]:
df_2000_0702 = pd.read_csv('/Users/giuseppe.martucci/Desktop/DecPartialBatch_Q/results/2000_0702.csv')
df_2000_0805 = pd.read_csv('/Users/giuseppe.martucci/Desktop/DecPartialBatch_Q/results/2000_0805.csv')


In [ ]:
df_ACGT0805 = pd.read_csv('/Users/giuseppe.martucci/Desktop/DecPartialBatch_Q/results/results_vect_paramsfixed_time2000_ACGT.csv')
df_ACGT0702 = pd.read_csv('/Users/giuseppe.martucci/Desktop/DecPartialBatch_Q/results/results_vect_paramsfixed_time2000_ACGT0702.csv')
df_ADGT0805 = pd.read_csv('/Users/giuseppe.martucci/Desktop/DecPartialBatch_Q/results/results_vect_paramsfixed_time2000_ADGT.csv')[:225]
df_ADGT0702 = pd.read_csv('/Users/giuseppe.martucci/Desktop/DecPartialBatch_Q/results/results_vect_paramsfixed_time2000_ADGT.csv')[225:]

import os
_RESULTS = '/Users/giuseppe.martucci/Desktop/DecPartialBatch_Q/results'
df_ACAD0702 = pd.read_csv(f'{_RESULTS}/results_vect_paramsfixed_time2000_ACAD0702.csv')
df_ACAD0805 = pd.read_csv(f'{_RESULTS}/results_vect_paramsfixed_time2000_ACAD0805.csv')

# ACGT with record_boundary=True (first-crossing columns for C_hat validation)
df_chat_0702 = pd.read_csv(f'{_RESULTS}/ACGT_Chat_validate_0702.csv')
df_chat_0805 = pd.read_csv(f'{_RESULTS}/ACGT_Chat_validate_0805.csv')


In [ ]:
def round_columns(df, columns, decimals):
    for column in columns:
        df[column] = df[column].round(decimals)
    return df

def round_to_quarters(df, columns, decimals):
    for column in columns:
        df[column] = (df[column]/ 0.25).round(decimals)* 0.25
    return df




In [ ]:
def filter_for_plot(df, na = 5, nl = 8):
    alphas = np.sort(df['alpha'].unique())
    selected = alphas[np.linspace(0, len(alphas) - 1, na).astype(int)]
    lambdas = np.sort(df['lambda'].unique())
    selected_l = lambdas[np.linspace(0, len(lambdas) - 1, nl).astype(int)]
    stric_s = selected_l[:-3]
    df_filtered = df[df['alpha'].isin(selected)]
    df_filtered = df_filtered[df_filtered['lambda'].isin(stric_s)]
    return df_filtered


In [ ]:
df_filtered = filter_for_plot(df_07_02, na = 5, nl = 8)
print('alpha values:',df_filtered['alpha'].unique())
print('lambda values:', df_filtered['lambda'].unique())


In [ ]:
df_filtered.M.value_counts()


## Uniform Initialization


In [ ]:
from basin import *


In [ ]:
#df_07_02_uniform_filtered = filter_for_plot(df_07_02_uniform, na = 5, nl = 8)
param_sets = [
    {'delta': 0.5, 'e': 0.05},
    {'delta': 0.5, 'e': 0.1},
    {'delta': 0.75, 'e': 0.05},
    {'delta': 0.75, 'e': 0.1},
    {'delta': 0.95, 'e': 0.05},
    {'delta': 0.95, 'e': 0.1},
]

y = 'P_GT'

t, r, p, s = 1, 0.7, 0.2, 0
theoretical_bounds = [get_aggregate_bounds_prop17(t=t, r=r, p=p, s=s, epsilon=ps['e'], delta=ps['delta']) for ps in param_sets]


plot_multiple_panels(df_2000_0702, param_sets, x_var="alpha", varying_var="lambda", y_var=y, m=2000, use_same_scale=True, cmap = 'viridis', theoretical_bounds=theoretical_bounds, theoretical_bounds_labels=("LB", "UB"), y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT})$")
plot_multiple_panels(df_2000_0702, param_sets, x_var="lambda", varying_var="alpha", y_var=y, m=2000, use_same_scale=True, cmap = 'viridis', theoretical_bounds=theoretical_bounds, theoretical_bounds_labels=("LB", "UB"), y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT})$")


In [ ]:
e, delta = 0.05, 0.95
t, r, p, s = 1, 0.8, 0.5, 0
get_aggregate_bounds(t=t, r=r, p=p, s=s, epsilon=e, d=delta, P_AD = True)


In [ ]:
#df_08_05_uniform_filtered = filter_for_plot(df_08_05_uniform, na = 5, nl = 8)
param_sets = [
    {'delta': 0.5, 'e': 0.05},
    {'delta': 0.5, 'e': 0.1},
    {'delta': 0.75, 'e': 0.05},
    {'delta': 0.75, 'e': 0.1},
    {'delta': 0.95, 'e': 0.05},
    {'delta': 0.95, 'e': 0.1},
]


y = 'P_GT'

t, r, p, s = 1, 0.8, 0.5, 0
theoretical_bounds = [get_aggregate_bounds_prop17(t=t, r=r, p=p, s=s, epsilon=ps['e'], delta=ps['delta']) for ps in param_sets]


plot_multiple_panels(df_2000_0805, param_sets, x_var="alpha", varying_var="lambda", y_var=y, m=2000, use_same_scale=True, cmap = 'cividis', theoretical_bounds=theoretical_bounds, theoretical_bounds_labels=("LB", "UB"), y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT})$")
plot_multiple_panels(df_2000_0805, param_sets, x_var="lambda", varying_var="alpha", y_var=y, m=2000, use_same_scale=True, cmap = 'cividis', theoretical_bounds=theoretical_bounds, theoretical_bounds_labels=("LB", "UB"), y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT})$")


## ADGT plots


In [ ]:
param_sets = [
    {'delta': 0.5, 'e': 0.05},
    {'delta': 0.5, 'e': 0.1},
    {'delta': 0.75, 'e': 0.05},
    {'delta': 0.75, 'e': 0.1},
    {'delta': 0.95, 'e': 0.05},
    {'delta': 0.95, 'e': 0.1},
]


y = 'P_GT'

t, r, p, s = 1, 0.8, 0.5, 0
from MultiAgent.basin import theoretical_P_AD_from_ADGT

theoretical_per_panel = [theoretical_P_GT_from_ADGT(t=t, r=r, p=p, s=s, delta=ps['delta'], e=ps['e']) for ps in param_sets]
theoretical_per_panel_c_s = [theoretical_P_GT_from_ADGT(t=t, r=r, p=p, s=s, delta=ps['delta'], e=ps['e']) for ps in param_sets]

plot_multiple_panels(df_ADGT0805, param_sets, x_var="alpha", varying_var="lambda", y_var=y, m=2000, use_same_scale=True, cmap = 'cividis', theoretical_basin_AD=theoretical_per_panel_c_s, y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AD,GT))$", theoretical_line_label=r"$\hat{B}$")
plot_multiple_panels(df_ADGT0805, param_sets, x_var="lambda", varying_var="alpha", y_var=y, m=2000, use_same_scale=True, cmap = 'cividis', theoretical_basin_AD=theoretical_per_panel_c_s, y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AD,GT))$", theoretical_line_label=r"$\hat{B}$")


In [ ]:
param_sets = [
    {'delta': 0.5, 'e': 0.05},
    {'delta': 0.5, 'e': 0.1},
    {'delta': 0.75, 'e': 0.05},
    {'delta': 0.75, 'e': 0.1},
    {'delta': 0.95, 'e': 0.05},
    {'delta': 0.95, 'e': 0.1},
]

m=2000
y = 'P_GT'
from MultiAgent.basin import theoretical_P_AD_from_ADGT
theoretical_per_panel_c = [theoretical_P_GT_from_ADGT(t=1, r=0.7, p=0.2, s=0, delta=ps['delta'], e=ps['e']) for ps in param_sets]

plot_multiple_panels(df_ADGT0702, param_sets, x_var="alpha", varying_var="lambda", y_var=y, m=m, use_same_scale=True, cmap = 'viridis', theoretical_basin_AD=theoretical_per_panel_c, y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AD,GT))$", theoretical_line_label=r"$\hat{B}$")
plot_multiple_panels(df_ADGT0702, param_sets, x_var="lambda", varying_var="alpha", y_var=y, m=m, use_same_scale=True, cmap = 'viridis', theoretical_basin_AD=theoretical_per_panel_c, y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AD,GT))$", theoretical_line_label=r"$\hat{B}$")


## ACGT Plots

Starting profile **(AC, GT)**. Player 1 can reach GT by crossing **A1D** first (direct) or by player 2 crossing **BC** first (indirect → profile **(AC, AD)**, may still reach GT later).

$\hat{C}$ (Prop. 17) is **only** the direct-route probability $\mathbb{P}[T_{\mathrm{A1D}} < T_{\mathrm{BC}}]$ in the $\alpha\to 0$ limit — **not** the probability of ending at (GT, GT).

### Three simulation columns (ACGT Chat validate CSVs only)

| Column | What it measures | Compare to $\hat{C}$? |
|--------|------------------|------------------------|
| `P_direct_GT_analytic` | Per run: $\mathbb{P}[T_{\mathrm{A1D}}<T_{\mathrm{BC}}]$ from **initial** $\Delta q$ (hitting-time formula). Panel value = mean over $M$ runs. | **Equality** — main theory check (should sit on the $\hat{C}$ line). |
| `P_direct_GT` | Per run: along the **simulated** Q-trajectory, did A1D cross before BC? (0/1). Panel value = fraction A1D-first. | **Equality** — tracks analytic at small $\alpha,\lambda$; uses discrete updates, not init-only math. |
| `P_GT` | Per run: did the run **end** at (GT, GT)? Includes direct **and** indirect (via (AC, AD)) successes. | **Inequality** — use band $[\hat{C}, 1]$, not a single line. Always $\hat{C} \le P_{\mathrm{GT}} \le 1$ in theory. |

**How to read the plots below (this section):** `y = P_GT` with reference band $[\hat{C}, 1]$. A gap **above** $\hat{C}$ is **expected** (indirect routes). Do **not** read that as theory failure.

### Which CSV for which plot?

| Dataframe | CSV | Has crossing columns? | Used for |
|-----------|-----|----------------------|----------|
| `df_ACGT0702` / `df_ACGT0805` | `results_vect_paramsfixed_time2000_ACGT*.csv` | No — only `P_GT` | **Plots below:** full BoA vs $[\hat{C},1]$ band |
| `df_chat_0702` / `df_chat_0805` | `ACGT_Chat_validate_*.csv` | Yes — `P_direct_GT`, `P_via_ACAD`, `P_direct_GT_analytic` | **Validate $\hat{C}$** section: equality tests vs $\hat{C}$ |

The Chat validate runs use `record_boundary=True`: the algorithm still runs to convergence, but the code **records** which boundary (A1D vs BC) was crossed first on each trajectory, and the analytic hitting-time probability at init.

**For a tight $\hat{C}$ match:** run the *Validate $\hat{B}$, $\hat{C}$, $\hat{D}$* cells — they use `df_chat_*`, not `df_ACGT_*`.


In [ ]:
from MultiAgent.basin import theoretical_P_AD_from_ADGT


In [ ]:
# Theory helpers are defined in the Proposition17 cell above (imports from basin.py).

In [ ]:
param_sets = [
    {'delta': 0.5, 'e': 0.05},
    {'delta': 0.5, 'e': 0.1},
    {'delta': 0.75, 'e': 0.05},
    {'delta': 0.75, 'e': 0.1},
    {'delta': 0.95, 'e': 0.05},
    {'delta': 0.95, 'e': 0.1},
]

y = 'P_GT'
t, r, p, s = 1, 0.8, 0.5, 0

# Latex reference (ACGT): C-hat <= BoA(pi^GT | (AC,GT)) <= 1
theoretical_bounds_acgt = [
    (theoretical_C_hat_from_ACGT(t=t, r=r, p=p, s=s, delta=ps['delta'], epsilon=ps['e']), 1.0)
    for ps in param_sets
]

plot_multiple_panels(
    df_ACGT0805, param_sets, x_var="alpha", varying_var="lambda", y_var=y,
    m=2000, use_same_scale=True, cmap='cividis',
    theoretical_bounds=theoretical_bounds_acgt,
    theoretical_bounds_labels=(r"$\hat{C}$", "UB"),
    y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AC,GT))$"
)
plot_multiple_panels(
    df_ACGT0805, param_sets, x_var="lambda", varying_var="alpha", y_var=y,
    m=2000, use_same_scale=True, cmap='cividis',
    theoretical_bounds=theoretical_bounds_acgt,
    theoretical_bounds_labels=(r"$\hat{C}$", "UB"),
    y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AC,GT))$"
)


## Boundary validation (Prop. 17)

Analytic basins $\hat B$, $\hat C$, $\hat D$ and aggregate LB/UB with simulation trajectories overlaid. Green/orange/purple shading is the **true** $\alpha\to 0$ boundary; trajectories should stay in the region determined by their initial point.

In [ ]:
param_sets = [
    {'delta': 0.5, 'e': 0.05},
    {'delta': 0.5, 'e': 0.1},
    {'delta': 0.75, 'e': 0.05},
    {'delta': 0.75, 'e': 0.1},
    {'delta': 0.95, 'e': 0.05},
    {'delta': 0.95, 'e': 0.1},
]

y = 'P_GT'
t, r, p, s = 1, 0.7, 0.2, 0

theoretical_bounds_acgt = [
    (theoretical_C_hat_from_ACGT(t=t, r=r, p=p, s=s, delta=ps['delta'], epsilon=ps['e']), 1.0)
    for ps in param_sets
]

plot_multiple_panels(
    df_ACGT0702, param_sets, x_var='alpha', varying_var='lambda', y_var=y,
    m=2000, use_same_scale=True, cmap='viridis',
    theoretical_bounds=theoretical_bounds_acgt,
    theoretical_bounds_labels=(r'$\hat{C}$', 'UB'),
    y_axis_label=r'$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AC,GT))$',
)
plot_multiple_panels(
    df_ACGT0702, param_sets, x_var='lambda', varying_var='alpha', y_var=y,
    m=2000, use_same_scale=True, cmap='viridis',
    theoretical_bounds=theoretical_bounds_acgt,
    theoretical_bounds_labels=(r'$\hat{C}$', 'UB'),
    y_axis_label=r'$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AC,GT))$',
)

theoretical_d_0702 = [
    (theoretical_D_hat_from_ACAD(t=t, r=r, p=p, s=s, delta=ps['delta'], e=ps['e']), 1.0)
    for ps in param_sets
]
plot_multiple_panels(
    df_ACAD0702, param_sets, x_var='alpha', varying_var='lambda', y_var=y, m=2000,
    use_same_scale=True, cmap='viridis', theoretical_bounds=theoretical_d_0702,
    theoretical_bounds_labels=(r'$\hat{D}$', None),
    y_axis_label=r'$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AC,AD))$',
)
plot_multiple_panels(
    df_ACAD0702, param_sets, x_var='lambda', varying_var='alpha', y_var=y, m=2000,
    use_same_scale=True, cmap='viridis', theoretical_bounds=theoretical_d_0702,
    theoretical_bounds_labels=(r'$\hat{D}$', None),
    y_axis_label=r'$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AC,AD))$',
)


In [ ]:
# Prop. 17 boundary validation: analytic shaded basins + simulation trajectories

def _run_one_trajectory(strategy1, strategy2, t, r, p, s, e, delta, alpha, lam, seed=0):
    np.random.seed(seed)
    algo = Algorithm(
        e=e, lambda_val=lam, alpha=alpha, delta=delta,
        init_mode='uniform', tolerance=1e-10, M=1,
        strategy1=strategy1, strategy2=strategy2,
        t=t, r=r, p=p, s=s, verbose=False,
    )
    q1h, q2h, _, _, _ = algo.run()
    return convert_dict_to_mean_df(q1h, mean=False)[0], convert_dict_to_mean_df(q2h, mean=False)[0]


def plot_prop17_boundaries_for_config(t, r, p, s, e, delta, alpha=0.01, lam=0.01, seed=7, label=''):
  b_hat = calculate_B_hat(t, r, p, s, e, delta)
  c_hat = calculate_C_hat(t, r, p, s, e, delta)
  d_hat = calculate_D_hat(t, r, p, s, e, delta, n_mc=150000)

  q1, q2 = _run_one_trajectory('AD', 'GT', t, r, p, s, e, delta, alpha, lam, seed=seed)
  fig, ax = plt.subplots(figsize=(6, 5))
  plot_adgt_boa_validation(
      t=t, r=r, p=p, s=s, epsilon=e, d=delta, q1=q1, q2=q2, alpha=alpha, ax=ax,
      title=rf'{label}$(AD,GT)$: $\hat{{B}}={b_hat:.3f}$',
  )
  chk = verify_trajectory_in_basin(q1, q2, 'ADGT', t, r, p, s, e, delta)
  print(f'{label}ADGT: stays in initial {chk["initial_basin"]} basin = {chk["stays_in_initial_basin"]}')
  plt.tight_layout(); plt.show()

  q1, q2 = _run_one_trajectory('AC', 'GT', t, r, p, s, e, delta, alpha, lam, seed=seed + 1)
  fig, ax = plot_acgt_boundary_validation(
      t=t, r=r, p=p, s=s, epsilon=e, d=delta, q1=q1, q2=q2, alpha=alpha,
      title=rf'{label}$(AC,GT)$: $\hat{{C}}={c_hat:.3f}$',
  )
  chk = verify_trajectory_in_basin(q1, q2, 'ACGT', t, r, p, s, e, delta)
  print(f'{label}ACGT: start in Ĉ region = {chk["initial_in_C_region"]:.0f}')
  plt.tight_layout(); plt.show()

  xh, yh = sample_acad_entry_points(t, r, p, s, e, delta, n=200, seed=seed + 2)
  fig, ax = plot_acad_entry_validation(
      t=t, r=r, p=p, s=s, epsilon=e, d=delta, entry_points=(xh, yh),
      title=rf'{label}$(AC,AD)\to(GT,AD)$: $\hat{{D}}={d_hat:.3f}$',
  )
  plt.tight_layout(); plt.show()

  lb, ub = get_aggregate_bounds_prop17(t, r, p, s, e, delta)
  print(f'{label}uniform aggregate BoA($\pi^{{GT}}$): LB={lb:.3f}, UB={ub:.3f}\n')


for cfg, name in [((1, 0.7, 0.2, 0), 'R=0.7,P=0.2 — '), ((1, 0.8, 0.5, 0), 'R=0.8,P=0.5 — ')]:
    t, r, p, s = cfg
    plot_prop17_boundaries_for_config(t, r, p, s, e=0.01, delta=0.6, alpha=0.01, lam=0.01, label=name)

In [ ]:
# Overlay aggregate LB/UB on uniform-init simulation panels (total boundaries)
param_sets = [
    {'delta': 0.5, 'e': 0.05},
    {'delta': 0.5, 'e': 0.1},
    {'delta': 0.75, 'e': 0.05},
    {'delta': 0.75, 'e': 0.1},
    {'delta': 0.95, 'e': 0.05},
    {'delta': 0.95, 'e': 0.1},
]

t, r, p, s = 1, 0.7, 0.2, 0
theoretical_bounds = [
    get_aggregate_bounds_prop17(t=t, r=r, p=p, s=s, epsilon=ps['e'], d=ps['delta'])
    for ps in param_sets
]

y = 'P_GT'
plot_multiple_panels(
    df_2000_0702, param_sets, x_var='alpha', varying_var='lambda', y_var=y, m=2000,
    use_same_scale=True, cmap='viridis', theoretical_bounds=theoretical_bounds,
    theoretical_bounds_labels=(r'LB', r'UB'),
    y_axis_label=r'$\mathrm{BoA}(\boldsymbol{\pi}^{GT})$',
)
plot_multiple_panels(
    df_2000_0702, param_sets, x_var='lambda', varying_var='alpha', y_var=y, m=2000,
    use_same_scale=True, cmap='viridis', theoretical_bounds=theoretical_bounds,
    theoretical_bounds_labels=(r'LB', r'UB'),
    y_axis_label=r'$\mathrm{BoA}(\boldsymbol{\pi}^{GT})$',
)


## Validate $\hat{B}$, $\hat{C}$, $\hat{D}$ against saved simulations

| Hat | Profile | CSV | Sim column |
|-----|---------|-----|------------|
| $\hat{B}$ | (AD, GT) | `results_vect_paramsfixed_time2000_ADGT.csv` | `P_GT` |
| $\hat{C}$ | (AC, GT) | `ACGT_Chat_validate_*.csv` | `P_direct_GT_analytic` |
| $\hat{D}$ | (AC, AD) | `results_vect_paramsfixed_time2000_ACAD*.csv` | `P_GT` |

### $\hat{C}$: `P_direct_GT_analytic` vs `P_direct_GT` vs `P_GT`

All three come from the same **(AC, GT)** runs with `record_boundary=True` (`ACGT_Chat_validate_*.csv`).

**The race.** From (AC, GT), the first profile change is whichever boundary is hit first:
- **A1D first** → direct route toward GT ($\hat{C}$ counts this).
- **BC first** → (AC, AD) first; the run may still end at (GT, GT) later ($\hat{C}$ does **not** count this).

---

**`P_direct_GT_analytic`** — *init-only, analytic*

- **Per run:** from starting Q-values, compute $\mathbb{P}[T_{\mathrm{A1D}} < T_{\mathrm{BC}}]$ (no trajectory).
- **Panel:** average over $M$ runs.
- **Plot:** vs $\hat{C}$ **line** (equality test).
- **Tests:** does our random init match the paper's uniform-on-$[-L,0]^2$ hitting-time integral?

---

**`P_direct_GT`** — *trajectory, empirical*

- **Per run:** watch the learning path; 1 if A1D crossed before BC, else 0.
- **Panel:** fraction A1D-first.
- **Plot:** vs $\hat{C}$ **line** (equality at small $\alpha,\lambda$).
- **Tests:** does finite-step learning preserve the same first-crossing order as the $\alpha\to 0$ flow?
- **vs analytic:** same question, but along discrete Q-updates; can differ slightly when $\alpha$ is large.

---

**`P_GT`** — *final outcome, full BoA*

- **Per run:** 1 if final profile is (GT, GT), else 0.
- **Panel:** fraction converging to (GT, GT).
- **Plot:** vs **band** $[\hat{C}, 1]$ (inequality), same as the main ACGT panels above.
- **Tests:** $\hat{C} \le \mathrm{BoA}(\pi^{GT}\mid AC,GT) \le 1$.
- **Gap above $\hat{C}$:** indirect successes after BC-first — **expected**, not a mismatch.

---

**Tiny example (100 runs).** 45 A1D-first; 40 of those end GT; 35 BC-first but still end GT → `P_direct_GT` $\approx 0.45 \approx \hat{C}$; `P_GT` $= 0.75$.

**Rule of thumb:** equality check → `P_direct_GT_analytic`; bound check → `P_GT` in $[\hat{C},1]$.


In [ ]:
# df_chat_0702 / df_chat_0805 loaded above (ACGT_Chat_validate_*.csv, record_boundary=True)
assert "P_direct_GT_analytic" in df_chat_0702.columns, "Re-run the data-loading cell for Chat validate CSVs"
print("Chat validate 0702 rows:", len(df_chat_0702))
print("Chat validate 0805 rows:", len(df_chat_0805))
df_chat_0702.head(3)


In [ ]:
from basin import add_hat_column, PAYOFF_PRESETS


def summarize_c_hat_validation(df, label, t, r, p, s):
    df = add_hat_column(df, "C_hat", t=t, r=r, p=p, s=s)
    df["err_analytic"] = df["P_direct_GT_analytic"] - df["C_hat"]
    df["err_direct"] = df["P_direct_GT"] - df["C_hat"]
    print(f"\n=== C_hat vs ACGT Chat {label} (R={r}, P={p}) ===")
    print(f"P_direct_GT_analytic: MAE={df['err_analytic'].abs().mean():.4f}")
    print(f"P_direct_GT trajectory: MAE={df['err_direct'].abs().mean():.4f}")
    display(df[["delta", "e", "alpha", "lambda", "C_hat", "P_direct_GT_analytic", "P_direct_GT", "P_GT"]].round(4))
    return df


def summarize_d_hat_validation(df, label, t, r, p, s, n_mc=50_000):
    df = add_hat_column(df, "D_hat", t=t, r=r, p=p, s=s, n_mc=n_mc)
    print(f"\n=== D_hat vs ACAD {label} (R={r}, P={p}) ===")
    print(f"P_GT vs D_hat: MAE={(df['P_GT'] - df['D_hat']).abs().mean():.4f}")
    display(df[["delta", "e", "alpha", "lambda", "D_hat", "P_GT"]].round(4))
    return df


def summarize_b_hat_validation(df, label, t, r, p, s):
    df = add_hat_column(df, "B_hat", t=t, r=r, p=p, s=s)
    print(f"\n=== B_hat vs ADGT {label} (R={r}, P={p}) ===")
    print(f"P_GT vs B_hat: MAE={(df['P_GT'] - df['B_hat']).abs().mean():.4f}")
    display(df[["delta", "e", "alpha", "lambda", "B_hat", "P_GT"]].round(4))
    return df


df_chat_0702 = summarize_c_hat_validation(df_chat_0702, "0702", **PAYOFF_PRESETS["0702"])
df_chat_0805 = summarize_c_hat_validation(df_chat_0805, "0805", **PAYOFF_PRESETS["0805"])
summarize_d_hat_validation(df_ACAD0702, "0702", **PAYOFF_PRESETS["0702"])
summarize_d_hat_validation(df_ACAD0805, "0805", **PAYOFF_PRESETS["0805"])
summarize_b_hat_validation(df_ADGT0702, "0702", **PAYOFF_PRESETS["0702"])
summarize_b_hat_validation(df_ADGT0805, "0805", **PAYOFF_PRESETS["0805"])


In [ ]:
write here the explanation between P_direct_GT_analytic, P_direct_GT, P_GT

In [ ]:
_ALL_PANELS_CHAT = [
    {"delta": 0.5, "e": 0.1},
    {"delta": 0.5, "e": 0.2},
    {"delta": 0.75, "e": 0.1},
    {"delta": 0.75, "e": 0.2},
    {"delta": 0.95, "e": 0.1},
    {"delta": 0.95, "e": 0.2},
]
_MAX_PANELS = 6  # plot_multiple_panels uses a 3x2 grid


def _panels_in_df(df):
    return [
        ps for ps in _ALL_PANELS_CHAT
        if not df[(df["delta"] == ps["delta"]) & (df["e"] == ps["e"])].empty
    ]


def _chunk(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


def plot_chat_validation(df, t, r, p, s, cmap, title_prefix):
    param_sets = _panels_in_df(df)
    m_chat = int(df["M"].iloc[0])

    for chunk_i, ps_chunk in enumerate(_chunk(param_sets, _MAX_PANELS)):
        chat_hat = [
            theoretical_C_hat_from_ACGT(t=t, r=r, p=p, s=s, delta=ps["delta"], epsilon=ps["e"])
            for ps in ps_chunk
        ]
        tag = f" {title_prefix}" if len(param_sets) <= _MAX_PANELS else f" {title_prefix} [{chunk_i + 1}]"

        for y_var, ylabel in [
            ("P_direct_GT", r"$P(\mathrm{A1D\ first})$"),
            ("P_direct_GT_analytic", r"$P(T_{A1D}<T_{BC})$ at init"),
        ]:
            plot_multiple_panels(
                df, ps_chunk, x_var="alpha", varying_var="lambda", y_var=y_var,
                m=m_chat, use_same_scale=True, cmap=cmap,
                theoretical_basin_AD=chat_hat, theoretical_line_label=r"$\hat{C}$",
                y_axis_label=rf"{tag}: {ylabel}",
            )
            plot_multiple_panels(
                df, ps_chunk, x_var="lambda", varying_var="alpha", y_var=y_var,
                m=m_chat, use_same_scale=True, cmap=cmap,
                theoretical_basin_AD=chat_hat, theoretical_line_label=r"$\hat{C}$",
                y_axis_label=rf"{tag}: {ylabel}",
            )

        chat_bounds = [(c, 1.0) for c in chat_hat]
        plot_multiple_panels(
            df, ps_chunk, x_var="alpha", varying_var="lambda", y_var="P_GT",
            m=m_chat, use_same_scale=True, cmap=cmap,
            theoretical_bounds=chat_bounds,
            theoretical_bounds_labels=(r"$\hat{C}$", "UB"),
            y_axis_label=rf"{tag}: $\\mathrm{{BoA}}(\\boldsymbol{{\\pi}}^{{GT}}\\mid(AC,GT))$",
        )


plot_chat_validation(df_chat_0702, t=1, r=0.7, p=0.2, s=0, cmap="viridis", title_prefix="0702")
plot_chat_validation(df_chat_0805, t=1, r=0.8, p=0.5, s=0, cmap="cividis", title_prefix="0805")


### Export appendix figures (`paper.tex` → `images/`)

Run after loading `df_ACGT*`, `df_ACAD*`, and `df_chat_*`. Writes filenames used in Appendix~\ref{app:acgt}.

In [ ]:
# Export appendix figures for paper.tex (Appendix~\ref{app:acgt})
import os

if os.path.basename(os.getcwd()) == "MultiAgent":
    _PAPER_IMAGES = os.path.abspath(os.path.join(os.getcwd(), "..", "images"))
else:
    _PAPER_IMAGES = os.path.abspath(os.path.join(os.getcwd(), "images"))

os.makedirs(_PAPER_IMAGES, exist_ok=True)

_PAPER_PARAM_SETS = [
    {"delta": 0.5, "e": 0.05},
    {"delta": 0.5, "e": 0.1},
    {"delta": 0.75, "e": 0.05},
    {"delta": 0.75, "e": 0.1},
    {"delta": 0.95, "e": 0.05},
    {"delta": 0.95, "e": 0.1},
]

_PLOT_KW = dict(
    param_sets=_PAPER_PARAM_SETS,
    x_var="alpha",
    varying_var="lambda",
    m=2000,
    use_same_scale=True,
    show=False,
)


def _save_acgt_boa(df, t, r, p, s, cmap, filename):
    chat_hat = [
        theoretical_C_hat_from_ACGT(t=t, r=r, p=p, s=s, delta=ps["delta"], epsilon=ps["e"])
        for ps in _PAPER_PARAM_SETS
    ]
    bounds = [(c, 1.0) for c in chat_hat]
    path = os.path.join(_PAPER_IMAGES, filename)
    plot_multiple_panels(
        df,
        y_var="P_GT",
        cmap=cmap,
        theoretical_bounds=bounds,
        theoretical_bounds_labels=(r"$\hat{C}$", "UB"),
        y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AC,GT))$",
        save_path=path,
        **_PLOT_KW,
    )
    print("saved", path)


def _save_acgt_boundary(df, t, r, p, s, cmap, filename):
    chat_hat = [
        theoretical_C_hat_from_ACGT(t=t, r=r, p=p, s=s, delta=ps["delta"], epsilon=ps["e"])
        for ps in _PAPER_PARAM_SETS
    ]
    path = os.path.join(_PAPER_IMAGES, filename)
    plot_multiple_panels(
        df,
        y_var="P_direct_GT_analytic",
        cmap=cmap,
        theoretical_basin_AD=chat_hat,
        theoretical_line_label=r"$\hat{C}$",
        y_axis_label=r"$P(T_{A1D}<T_{BC})$ at init",
        save_path=path,
        **_PLOT_KW,
    )
    print("saved", path)


def _save_acad_boa(df, t, r, p, s, cmap, filename):
    d_hat = [
        theoretical_D_hat_from_ACAD(t=t, r=r, p=p, s=s, delta=ps["delta"], e=ps["e"])
        for ps in _PAPER_PARAM_SETS
    ]
    path = os.path.join(_PAPER_IMAGES, filename)
    plot_multiple_panels(
        df,
        y_var="P_GT",
        cmap=cmap,
        theoretical_basin_AD=d_hat,
        theoretical_line_label=r"$\hat{D}$",
        y_axis_label=r"$\mathrm{BoA}(\boldsymbol{\pi}^{GT}\mid(AC,AD))$",
        save_path=path,
        **_PLOT_KW,
    )
    print("saved", path)


# (AC,GT) full BoA vs C_hat
_save_acgt_boa(df_ACGT0702, t=1, r=0.7, p=0.2, s=0, cmap="viridis", filename="af0702_acgt.png")
_save_acgt_boa(df_ACGT0805, t=1, r=0.8, p=0.5, s=0, cmap="cividis", filename="af_0805_acgt.png")

# (AC,GT) first-crossing validation of C_hat
_save_acgt_boundary(df_chat_0702, t=1, r=0.7, p=0.2, s=0, cmap="viridis", filename="af0702_acgt_boundary.png")
_save_acgt_boundary(df_chat_0805, t=1, r=0.8, p=0.5, s=0, cmap="cividis", filename="af0805_acgt_boundary.png")

# (AC,AD) BoA vs D_hat
_save_acad_boa(df_ACAD0702, t=1, r=0.7, p=0.2, s=0, cmap="viridis", filename="af0702_acad.png")
_save_acad_boa(df_ACAD0805, t=1, r=0.8, p=0.5, s=0, cmap="cividis", filename="af0805_acad.png")

# Entry plane for (AC,AD) -> (GT,AD)  (Figure~\ref{fig:acad-entry})
_xh, _yh = sample_acad_entry_points(t=1, r=0.7, p=0.2, s=0, epsilon=0.01, d=0.6, n=300, seed=7)
_fig_entry, _ax_entry = plot_acad_entry_validation(
    t=1, r=0.7, p=0.2, s=0, epsilon=0.01, d=0.6,
    entry_points=(_xh, _yh),
    title=r"$(AC,AD)\to(GT,AD)$ entry plane",
)
_entry_path = os.path.join(_PAPER_IMAGES, "acad_entry_plane.png")
_fig_entry.savefig(_entry_path, dpi=300, bbox_inches="tight")
plt.close(_fig_entry)
print("saved", _entry_path)